# Tutorial 1: Basic Annotation with mbanno

This notebook demonstrates the basic annotation workflow using mbanno.

**Prerequisites**:
- mbanno installed (`pip install -e .`)
- Reference data downloaded (`mbanno download-reference`)
- A query AnnData (h5ad) file with mouse brain single-cell data

## 1. Setup

Import and check environment.

In [ ]:
import scanpy as sc
import mbanno
from mbanno import ConsensusAnnotator, QCProcessor

print(f"mbanno version: {mbanno.__version__}")
print(f"scanpy version: {sc.__version__}")

## 2. Load Data

Load your mouse brain single-cell data.

In [ ]:
# Replace with your data path
adata = sc.read_h5ad("data/query.h5ad")
print(f"Cells: {adata.n_obs:,}")
print(f"Genes: {adata.n_vars:,}")

## 3. Quality Control

Run QC filtering: doublet detection, MT filtering, normalization.

In [ ]:
qc = QCProcessor(min_genes=200, min_counts=500, max_pct_mt=20.0)
adata_qc = qc.run(adata)

summary = qc.get_summary(adata_qc)
for k, v in summary.items():
    print(f"  {k}: {v}")

## 4. Consensus Annotation

Run multi-tool consensus annotation.

In [ ]:
annotator = ConsensusAnnotator(
    reference="allen-consensus-wmb",
    taxonomy_level="subclass"
)

results = annotator.annotate(
    adata_qc,
    methods=["marker"],  # Add more: "mapmycells", "scanvi", "celltypist"
    min_confidence=0.5,
)

## 5. Examine Results

Review annotations and confidence.

In [ ]:
annotations = results["annotations"]
print(annotations.head())
print(f"\nAnnotation columns: {list(annotations.columns)}")

In [ ]:
# Confidence distribution
confidence = results["confidence"]
print(f"Mean confidence: {confidence['confidence'].mean():.3f}")
print(confidence["confidence_level"].value_counts())

## 6. Save and Report

Save results and generate report.

In [ ]:
from pathlib import Path
output_dir = Path("results/tutorial_01")

annotator.save_results(results, output_dir)
annotator.generate_report(results, output_dir)
print(f"Results saved to: {output_dir.resolve()}")

## Next Steps

- Tutorial 2: Multi-tool cross-validation
- Tutorial 3: Spatial data annotation
- Tutorial 4: Disease sample annotation

**Cite**: Yao, Z. et al. *Nature* 624, 317-332 (2023). doi:10.1038/s41586-023-06812-z